# Snowball or Comeback? How Early Objectives Shape Professional League of Legends

**Name(s)**: (your name here)

**Website Link**: https://harsh5559.github.io/259_FINAL/

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go
pd.options.plotting.backend = 'plotly'

from dsc259r_utils import *

## Step 1: Introduction

In [ ]:
# Load the full 2022 Oracle's Elixir dataset.
# low_memory=False suppresses the mixed-type warning on the 'url' column.
df = pd.read_csv(
    '2022_LoL_esports_match_data_from_OraclesElixir.csv',
    low_memory=False
)

# Each game produces 12 rows: one per player (10 total) and one per team (2 total).
# We keep only the team-level summary rows for our analysis.
teams = df[df['position'] == 'team'].copy()

print(f'Raw dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Team-level rows: {teams.shape[0]:,} rows ({teams["gameid"].nunique():,} unique games)')
print(f'Leagues represented: {teams["league"].nunique()}')
print(f'Date range: {teams["date"].min()} → {teams["date"].max()}')

### Research Question

**In professional League of Legends, do early-game objectives provide a genuine strategic advantage independent of a team's overall gold position, and how reliably can match outcomes be predicted from the first 15 minutes of play?**

In competitive LoL, players and analysts often debate whether objectives like *first dragon* are truly impactful or merely symptoms of a team that is already ahead. A team winning fights and controlling the map will naturally secure dragons, towers, and first blood — but do those objectives *independently* boost win probability, or are they just correlated with a gold lead that was going to win the game anyway?

We use the 2022 Oracle's Elixir dataset — 12,415 professional matches across 55 regions — to disentangle the effect of individual early objectives from overall early-game dominance. Specifically, we examine whether first dragon provides a measurable competitive edge even in closely-contested games (where gold differentials at 15 minutes are small), and whether stacking multiple early objectives creates compounding or merely additive effects on win probability.

**Relevant columns:**

| Column | Description |
|--------|-------------|
| `result` | Match outcome: `1` = win, `0` = loss |
| `golddiffat15` | Team's gold lead/deficit at 15 minutes vs. opponent |
| `xpdiffat15` | Team's XP lead/deficit at 15 minutes |
| `csdiffat15` | Team's CS lead/deficit at 15 minutes |
| `firstblood` | Whether this team secured first blood (`1`/`0`) |
| `firstdragon` | Whether this team claimed the first dragon |
| `firsttower` | Whether this team destroyed the first tower |
| `firstbaron` | Whether this team claimed the first baron |
| `gamelength` | Total game duration in seconds |
| `side` | Which side a team played on: `Blue` or `Red` |
| `league` | Tournament/region identifier (e.g. LCK, LPL, LEC) |
| `kills` | Total team kills |
| `deaths` | Total team deaths |

In [ ]:
# Preview the relevant columns
relevant_cols = [
    'gameid', 'league', 'side', 'teamname', 'result',
    'gamelength', 'firstblood', 'firstdragon', 'firstbaron',
    'golddiffat15', 'xpdiffat15', 'csdiffat15', 'kills', 'deaths'
]
display_df(teams[relevant_cols])

## Step 2: Data Cleaning and Exploratory Data Analysis

In [ ]:
# --- Data Cleaning ---

# 1. Parse date strings into proper datetime objects so we can do time-based operations.
teams['date'] = pd.to_datetime(teams['date'])

# 2. Convert gamelength from seconds to minutes for readability.
teams['gamelength_min'] = teams['gamelength'] / 60

# 3. The 'result' column is already 0/1 integers — convert to bool for semantic clarity.
teams['result'] = teams['result'].astype(bool)

# 4. Assess missingness in the early-game diff columns.
#    Games shorter than ~15 minutes will have NaN for golddiffat15 etc.
early_cols = ['golddiffat15', 'xpdiffat15', 'csdiffat15']
for col in early_cols:
    n_missing = teams[col].isna().sum()
    print(f'{col}: {n_missing} missing ({n_missing / len(teams):.1%})')

# Create a clean subset for early-game analysis (rows where 15-min data exists).
teams_15 = teams.dropna(subset=early_cols).copy()
print(f'\nRows with complete 15-min data: {len(teams_15):,} of {len(teams):,}')

# 5. Count how many early objectives (first blood, first dragon, first tower) each team secured.
obj_cols = ['firstblood', 'firstdragon', 'firsttower']
teams['early_obj_count'] = teams[obj_cols].sum(axis=1).astype(int)
teams_15['early_obj_count'] = teams_15[obj_cols].sum(axis=1).astype(int)

In [ ]:
# Preview the cleaned DataFrame
display_df(teams[relevant_cols])

### Univariate Analysis

In [ ]:
# Distribution of game length (in minutes)
fig_gamelength = px.histogram(
    teams,
    x='gamelength_min',
    nbins=60,
    title='Distribution of Professional Game Lengths (2022)',
    labels={'gamelength_min': 'Game Length (minutes)', 'count': 'Number of Games'},
    template='dsc259r'
)
fig_gamelength.update_layout(
    xaxis_title='Game Length (minutes)',
    yaxis_title='Number of Games',
    bargap=0.05
)
fig_gamelength.write_html('assets/gamelength_dist.html', include_plotlyjs='cdn')
fig_gamelength.show()

In [ ]:
# Win rate by number of early objectives secured (first blood, first dragon, first tower)
obj_wr = (
    teams.groupby('early_obj_count')['result']
    .agg(win_rate='mean', games='count')
    .reset_index()
)
obj_wr['win_rate_pct'] = (obj_wr['win_rate'] * 100).round(1)

fig_obj = px.bar(
    obj_wr,
    x='early_obj_count',
    y='win_rate_pct',
    text='win_rate_pct',
    title='Win Rate by Number of Early Objectives Secured (FB + Dragon + Tower)',
    labels={'early_obj_count': 'Early Objectives Secured (out of 3)',
            'win_rate_pct': 'Win Rate (%)'},
    color='win_rate_pct',
    color_continuous_scale=['#D62728', '#FF7F0E', '#2CA02C', '#1B7A2B'],
    template='dsc259r'
)
fig_obj.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig_obj.update_layout(
    yaxis_range=[0, 95],
    xaxis_title='Number of Early Objectives Secured',
    yaxis_title='Win Rate (%)',
    coloraxis_showscale=False,
    xaxis=dict(tickmode='linear')
)
fig_obj.write_html('assets/objective_stacking.html', include_plotlyjs='cdn')
fig_obj.show()

print(obj_wr[['early_obj_count', 'win_rate_pct', 'games']].to_string(index=False))

### Bivariate Analysis

In [ ]:
# Gold differential at 15 minutes by match outcome
teams_15_plot = teams_15.copy()
teams_15_plot['Outcome'] = teams_15_plot['result'].map({True: 'Win', False: 'Loss'})

fig_gold = px.box(
    teams_15_plot,
    x='Outcome',
    y='golddiffat15',
    color='Outcome',
    color_discrete_map={'Win': '#2CA02C', 'Loss': '#D62728'},
    title='Gold Differential at 15 Minutes by Match Outcome',
    labels={'golddiffat15': 'Gold Differential at 15 min', 'Outcome': 'Match Outcome'},
    template='dsc259r'
)
fig_gold.update_layout(
    yaxis_title='Gold Differential at 15 min',
    xaxis_title='Match Outcome',
    showlegend=False
)
fig_gold.add_hline(y=0, line_dash='dash', line_color='gray', opacity=0.5)
fig_gold.write_html('assets/golddiff_by_outcome.html', include_plotlyjs='cdn')
fig_gold.show()

In [ ]:
# Scatter: gold diff at 15 vs total kills, colored by outcome
fig_scatter = px.scatter(
    teams_15_plot.sample(3000, random_state=42),  # sample for clarity
    x='golddiffat15',
    y='kills',
    color='Outcome',
    color_discrete_map={'Win': '#2CA02C', 'Loss': '#D62728'},
    title='Gold Lead at 15 Min vs. Total Kills (sample of 3,000 games)',
    labels={'golddiffat15': 'Gold Diff at 15 min', 'kills': 'Total Kills'},
    opacity=0.4,
    template='dsc259r'
)
fig_scatter.update_layout(
    xaxis_title='Gold Differential at 15 min',
    yaxis_title='Total Team Kills'
)
fig_scatter.show()

### Interesting Aggregates

In [ ]:
# Aggregate: win rate when each early objective is secured vs. not — by top leagues
top_leagues = teams.groupby('league')['gameid'].nunique().nlargest(15).index
top_teams = teams[teams['league'].isin(top_leagues)]

obj_impact = []
for obj in ['firstblood', 'firstdragon', 'firsttower']:
    for league in top_leagues:
        sub = top_teams[top_teams['league'] == league]
        got = sub[sub[obj] == 1]['result'].mean() * 100
        not_got = sub[sub[obj] == 0]['result'].mean() * 100
        obj_impact.append({
            'league': league,
            'objective': obj.replace('first', 'First ').title(),
            'Win Rate With (%)': round(got, 1),
            'Win Rate Without (%)': round(not_got, 1),
            'Advantage (pp)': round(got - not_got, 1)
        })

impact_df = pd.DataFrame(obj_impact)
pivot_impact = impact_df.pivot_table(
    index='league',
    columns='objective',
    values='Advantage (pp)'
)
pivot_impact = pivot_impact.sort_values(pivot_impact.columns[0], ascending=False)
display_df(pivot_impact)

In [ ]:
# Key aggregate: first dragon win rate in "close" vs "decisive" games at 15 min
# This directly motivates the hypothesis test — does dragon matter independently of gold state?
close = teams_15[teams_15['golddiffat15'].abs() <= 1000]
decisive = teams_15[teams_15['golddiffat15'].abs() > 3000]

rows = []
for label, subset in [('Close (|gold diff| ≤ 1k)', close), ('Decisive (|gold diff| > 3k)', decisive)]:
    for fd_val, fd_label in [(1, 'Got First Dragon'), (0, 'No First Dragon')]:
        s = subset[subset['firstdragon'] == fd_val]
        rows.append({
            'Game Type': label,
            'First Dragon': fd_label,
            'Win Rate (%)': round(s['result'].mean() * 100, 1),
            'Games': len(s)
        })

dragon_table = pd.DataFrame(rows)
print('First Dragon Impact by Game Closeness at 15 Minutes:')
print(dragon_table.to_markdown(index=False))

---
## Checkpoint 1 Answers

### (1) Dataset Choice
We chose the **League of Legends** dataset (2022 Oracle's Elixir). It covers 12,415 professional matches across 55 regions worldwide, with detailed early-game state data (gold/XP/CS differentials at 10, 15, 20, and 25 minutes) and objective flags (first blood, first dragon, first tower, first baron). This makes it ideal for investigating whether individual objectives have an independent impact on match outcomes or are merely correlated with overall dominance.

### (2) Plotly Visualization
*See the "Win Rate by Number of Early Objectives Secured" bar chart above — screenshot that plot for the Gradescope submission.*

### (3) Hypothesis Test Plan (Step 4)
Players and analysts debate whether first dragon is a truly impactful objective or merely a symptom of an already-winning team. We test this by conditioning on closely-contested games.

- **Null Hypothesis**: Among games where the gold differential at 15 minutes is within ±1,000 gold (i.e. closely contested), teams that secured first dragon win at the same rate as teams that did not. Any observed difference is due to random chance.
- **Alternative Hypothesis**: Teams that secured first dragon win at a significantly higher rate even in closely-contested games, indicating that first dragon provides a genuine strategic advantage independent of gold state.
- **Test Statistic**: Difference in win rates: Win Rate(first dragon, close game) − Win Rate(no first dragon, close game).
- **Method**: Permutation test — among the ~5,700 close-game rows, shuffle the `firstdragon` labels 10,000 times and recompute the difference in win rates to build the null distribution.

### (4) Prediction Problem Plan (Steps 5–8)
- **Column to predict**: `result` (1 = win, 0 = loss)
- **Type**: Binary classification
- **Framing**: Predict match outcome using only information available at the 15-minute mark: `golddiffat15`, `xpdiffat15`, `csdiffat15`, `firstblood`, `firstdragon`, `firsttower`, `side`.
- **Evaluation metric**: F1-score (appropriate for balanced binary classification; captures both precision and recall).
---

## Step 3: Assessment of Missingness

### NMAR Analysis

We believe that `golddiffat15` — the column with the most notable missingness (15.2% of team rows) — is **not NMAR**. Its missingness is almost entirely explained by `gamelength`: games that ended before the 15-minute mark simply have no 15-minute statistics to record. Since game duration is observed, this makes the missingness **MAR** (Missing At Random), conditional on game length.

A column that could plausibly be **NMAR** is `url` (the match replay link). Whether a replay URL is recorded might depend on factors intrinsic to the match itself — for instance, replays from games with technical issues, controversial rulings, or server instability may be less likely to be archived. Since these factors relate to the *content* of the missing data (the replay) rather than other observed columns, this would constitute NMAR. Obtaining additional data about tournament broadcast and archival policies could potentially explain the missingness pattern and reclassify it as MAR.

### Missingness Dependency

We analyze whether the missingness of `golddiffat15` depends on other columns using permutation tests at the α = 0.05 significance level.

In [ ]:
# Missingness permutation tests for golddiffat15

teams['gd15_missing'] = teams['golddiffat15'].isna()
print(f'golddiffat15 missing: {teams["gd15_missing"].sum():,} / {len(teams):,} '
      f'({teams["gd15_missing"].mean():.1%})')

rng_miss = np.random.default_rng(42)
N_PERM_MISS = 10_000

# --- Test 1: Missingness DEPENDS on gamelength ---
# Games that ended before 15 min have no 15-min stats.
gl_missing = teams.loc[teams['gd15_missing'], 'gamelength_min'].values
gl_not_missing = teams.loc[~teams['gd15_missing'], 'gamelength_min'].values
obs_diff_gl = gl_missing.mean() - gl_not_missing.mean()

gl_all = teams['gamelength_min'].values
miss_labels = teams['gd15_missing'].values.copy()
perm_diffs_gl = np.empty(N_PERM_MISS)
for i in range(N_PERM_MISS):
    rng_miss.shuffle(miss_labels)
    perm_diffs_gl[i] = gl_all[miss_labels].mean() - gl_all[~miss_labels].mean()

pval_gl = np.mean(np.abs(perm_diffs_gl) >= np.abs(obs_diff_gl))

print('\n--- Test 1: Missingness vs. Game Length ---')
print(f'Mean game length (missing):     {gl_missing.mean():.2f} min')
print(f'Mean game length (not missing): {gl_not_missing.mean():.2f} min')
print(f'Observed difference in means:   {obs_diff_gl:.4f} min')
print(f'p-value: {pval_gl}')
print('Conclusion: Reject H0 — missingness of golddiffat15 DEPENDS on gamelength.')

# Visualization: overlaid histograms
fig_miss_gl = go.Figure()
fig_miss_gl.add_trace(go.Histogram(
    x=gl_not_missing, name='Not Missing', opacity=0.6,
    marker_color='#2CA02C', histnorm='probability', nbinsx=50))
fig_miss_gl.add_trace(go.Histogram(
    x=gl_missing, name='Missing', opacity=0.6,
    marker_color='#D62728', histnorm='probability', nbinsx=50))
fig_miss_gl.update_layout(
    title='Game Length Distribution: golddiffat15 Missing vs Not Missing',
    xaxis_title='Game Length (minutes)', yaxis_title='Proportion',
    barmode='overlay', template='dsc259r')
fig_miss_gl.write_html('assets/missingness_gamelength.html', include_plotlyjs='cdn')
fig_miss_gl.show()

# --- Test 2: Missingness does NOT depend on side ---
# Both teams in a game share the same game duration, so side should not affect missingness.
blue_miss_rate = teams.loc[teams['side'] == 'Blue', 'gd15_missing'].mean()
red_miss_rate = teams.loc[teams['side'] == 'Red', 'gd15_missing'].mean()
obs_diff_side = blue_miss_rate - red_miss_rate

side_arr = teams['side'].values.copy()
gd_miss_arr = teams['gd15_missing'].values
perm_diffs_side = np.empty(N_PERM_MISS)
for i in range(N_PERM_MISS):
    rng_miss.shuffle(side_arr)
    blue_mask = side_arr == 'Blue'
    perm_diffs_side[i] = gd_miss_arr[blue_mask].mean() - gd_miss_arr[~blue_mask].mean()

pval_side = np.mean(np.abs(perm_diffs_side) >= np.abs(obs_diff_side))

print('\n--- Test 2: Missingness vs. Side ---')
print(f'Blue side missing rate: {blue_miss_rate:.4f}')
print(f'Red side missing rate:  {red_miss_rate:.4f}')
print(f'Observed difference:    {obs_diff_side:.6f}')
print(f'p-value: {pval_side}')
print('Conclusion: Fail to reject H0 — missingness of golddiffat15 does NOT depend on side.')

# Visualization: permutation distribution with observed line
fig_miss_side = go.Figure()
fig_miss_side.add_trace(go.Histogram(
    x=perm_diffs_side, marker_color='#1f77b4', opacity=0.7, nbinsx=50,
    name='Permutation Distribution'))
fig_miss_side.add_vline(
    x=obs_diff_side, line_dash='dash', line_color='red',
    annotation_text=f'Observed = {obs_diff_side:.4f}')
fig_miss_side.update_layout(
    title='Permutation Test: golddiffat15 Missingness vs Side',
    xaxis_title='Difference in Missing Rate (Blue − Red)',
    yaxis_title='Count', template='dsc259r')
fig_miss_side.write_html('assets/missingness_side_perm.html', include_plotlyjs='cdn')
fig_miss_side.show()

## Step 4: Hypothesis Testing

In [ ]:
# Step 4: Hypothesis Testing (two tests required for Checkpoint 2)

rng = np.random.default_rng(42)
N_PERM = 10_000

# --------------------
# Hypothesis Test 1
# --------------------
# Question: In close games (|gold diff at 15| <= 1000), does first dragon increase win rate?
close_games = teams_15[teams_15['golddiffat15'].abs() <= 1000].copy()

obs_stat_1 = (
    close_games.loc[close_games['firstdragon'] == 1, 'result'].mean()
    - close_games.loc[close_games['firstdragon'] == 0, 'result'].mean()
)

perm_stats_1 = np.empty(N_PERM)
fd_labels = close_games['firstdragon'].to_numpy().copy()
outcomes = close_games['result'].astype(int).to_numpy()

for i in range(N_PERM):
    rng.shuffle(fd_labels)
    perm_stats_1[i] = outcomes[fd_labels == 1].mean() - outcomes[fd_labels == 0].mean()

pval_1 = np.mean(perm_stats_1 >= obs_stat_1)

print('Hypothesis Test 1: First Dragon Effect in Close Games')
print('H0: In close games, first-dragon and non-first-dragon teams have equal win rates.')
print('HA: In close games, first-dragon teams have higher win rates.')
print(f'Observed statistic (win-rate difference): {obs_stat_1:.4f} ({obs_stat_1*100:.2f} pp)')
print(f'p-value: {pval_1:.5f} (report as < {1/N_PERM:.4f} when this is 0.00000)')
print()

# --------------------
# Hypothesis Test 2
# --------------------
# Question: Do games with smaller 15-min gold gaps last longer than games with larger gaps?
teams_15['abs_gd15'] = teams_15['golddiffat15'].abs()
q25 = teams_15['abs_gd15'].quantile(0.25)
q75 = teams_15['abs_gd15'].quantile(0.75)

length_groups = teams_15[(teams_15['abs_gd15'] <= q25) | (teams_15['abs_gd15'] >= q75)].copy()
length_groups['gap_group'] = np.where(length_groups['abs_gd15'] >= q75, 'large', 'small')

obs_stat_2 = (
    length_groups.loc[length_groups['gap_group'] == 'small', 'gamelength_min'].mean()
    - length_groups.loc[length_groups['gap_group'] == 'large', 'gamelength_min'].mean()
)

perm_stats_2 = np.empty(N_PERM)
gap_labels = length_groups['gap_group'].to_numpy().copy()
lengths = length_groups['gamelength_min'].to_numpy()

for i in range(N_PERM):
    rng.shuffle(gap_labels)
    perm_stats_2[i] = lengths[gap_labels == 'small'].mean() - lengths[gap_labels == 'large'].mean()

pval_2 = np.mean(perm_stats_2 >= obs_stat_2)

print('Hypothesis Test 2: Early Gold Gap vs Game Length')
print('H0: Mean game length is the same for small-gap and large-gap games.')
print('HA: Small-gap games have longer mean game length than large-gap games.')
print(f'Q1 threshold: <= {q25:.0f} gold, Q3 threshold: >= {q75:.0f} gold')
print(f'Observed statistic (minutes): {obs_stat_2:.4f}')
print(f'p-value: {pval_2:.5f} (report as < {1/N_PERM:.4f} when this is 0.00000)')

# Save key results for easy copy into website/checkpoint answers
checkpoint2_hypothesis_results = {
    'test_1_obs_pp': round(obs_stat_1 * 100, 2),
    'test_1_p_value': pval_1,
    'test_2_obs_minutes': round(obs_stat_2, 2),
    'test_2_p_value': pval_2,
}
print('\nSummary:', checkpoint2_hypothesis_results)

## Step 5: Framing a Prediction Problem

In [ ]:
### Prediction Problem Framing

We frame a **binary classification** problem:

- **Response variable**: `result` (`1` = win, `0` = loss)
- **Time of prediction**: exactly **15 minutes** into the game
- **Allowed features** (known by minute 15):
  - `golddiffat15`, `xpdiffat15`, `csdiffat15`
  - `firstblood`, `firstdragon`, `firsttower`
  - `side`

Why this framing is valid: every predictor above is observable at or before minute 15, so the model does not leak future information.

Primary metric: **F1-score** (with accuracy, precision, and recall also reported). F1 balances precision and recall and is useful when we care about both false positives and false negatives.

## Step 6: Baseline Model

In [ ]:
# Step 6: Baseline Model
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

baseline_features = [
    'golddiffat15', 'xpdiffat15', 'csdiffat15',
    'firstblood', 'firstdragon', 'firsttower', 'side'
]

model_df = teams_15[baseline_features + ['result']].copy()
X = model_df[baseline_features]
y = model_df['result'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_cols = ['golddiffat15', 'xpdiffat15', 'csdiffat15']
nominal_cols = ['side']
binary_cols = ['firstblood', 'firstdragon', 'firsttower']

baseline_preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median'))
        ]), numeric_cols),
        ('nom', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(drop='if_binary', handle_unknown='ignore'))
        ]), nominal_cols),
        ('bin', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent'))
        ]), binary_cols),
    ]
)

baseline_model = Pipeline([
    ('preprocess', baseline_preprocessor),
    ('clf', LogisticRegression(max_iter=2000))
])

baseline_model.fit(X_train, y_train)
y_pred = baseline_model.predict(X_test)

baseline_metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'F1': f1_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
}

print('Baseline model: Logistic Regression + preprocessing pipeline')
print(f'Train size: {len(X_train):,} | Test size: {len(X_test):,}')
for metric, value in baseline_metrics.items():
    print(f'{metric}: {value:.4f}')

## Step 7: Final Model

In [ ]:
### Planned Improvements for Final Model

To improve on the baseline model, we will:

1. **Engineer additional features**
   - `abs_golddiffat15` (magnitude of lead regardless of side)
   - `early_obj_count` (sum of first blood + first dragon + first tower)
   - interaction terms like `golddiffat15 × firstdragon`

2. **Tune hyperparameters using GridSearchCV**
   - For logistic regression: `C`, penalty type, class weights
   - For tree-based alternatives (e.g., Random Forest / Gradient Boosting): depth, min samples split, number of estimators

3. **Compare model families**
   - Keep logistic regression as interpretable baseline
   - Evaluate nonlinear models that can capture threshold effects (e.g., a 2k gold lead may change win odds nonlinearly)

4. **Select final model by validation performance**
   - Primary metric: F1
   - Secondary checks: precision/recall balance and calibration behavior

The goal is not just higher score, but a model that better reflects how pro games snowball from early leads and objectives.

In [ ]:
# Step 7: Final Model — enhanced Logistic Regression with engineered features

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Engineer new features from the data (all observable at 15 min)
teams_15['abs_golddiffat15'] = teams_15['golddiffat15'].abs()
teams_15['gold_xp_interaction'] = teams_15['golddiffat15'] * teams_15['xpdiffat15']

final_features = [
    'golddiffat15', 'xpdiffat15', 'csdiffat15',
    'firstblood', 'firstdragon', 'firsttower', 'side',
    'abs_golddiffat15', 'early_obj_count', 'gold_xp_interaction'
]

# Use the same train/test indices from baseline for fair comparison
train_idx = X_train.index
test_idx = X_test.index

X_train_f = teams_15.loc[train_idx, final_features]
X_test_f = teams_15.loc[test_idx, final_features]
y_train_f = teams_15.loc[train_idx, 'result'].astype(int)
y_test_f = teams_15.loc[test_idx, 'result'].astype(int)

numeric_final = ['golddiffat15', 'xpdiffat15', 'csdiffat15',
                 'abs_golddiffat15', 'gold_xp_interaction']
binary_final = ['firstblood', 'firstdragon', 'firsttower', 'early_obj_count']
nominal_final = ['side']

final_preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_final),
        ('nom', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(drop='if_binary', handle_unknown='ignore'))
        ]), nominal_final),
        ('bin', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent'))
        ]), binary_final),
    ]
)

# Hyperparameters to tune: regularization strength C controls the bias-variance tradeoff.
# Stronger regularization (lower C) prevents overfitting to noisy features;
# weaker regularization (higher C) lets the model exploit subtle patterns in
# the engineered interaction/magnitude features.
final_pipeline = Pipeline([
    ('preprocess', final_preprocessor),
    ('clf', LogisticRegression(max_iter=2000))
])

param_grid_lr = {
    'clf__C': [0.01, 0.1, 1, 10, 100],
    'clf__penalty': ['l2'],
}

grid_search = GridSearchCV(
    final_pipeline, param_grid_lr, cv=5, scoring='f1', n_jobs=-1
)
grid_search.fit(X_train_f, y_train_f)

print(f'Best hyperparameters: {grid_search.best_params_}')
print(f'Best cross-validation F1: {grid_search.best_score_:.4f}')

# Evaluate on the held-out test set
final_model = grid_search.best_estimator_
y_pred_final = final_model.predict(X_test_f)

final_metrics = {
    'Accuracy': accuracy_score(y_test_f, y_pred_final),
    'F1': f1_score(y_test_f, y_pred_final),
    'Precision': precision_score(y_test_f, y_pred_final),
    'Recall': recall_score(y_test_f, y_pred_final),
}

print('\nFinal Model Performance (test set):')
for metric, value in final_metrics.items():
    diff = value - baseline_metrics[metric]
    print(f'  {metric}: {value:.4f}  ({"+" if diff >= 0 else ""}{diff:.4f} vs baseline)')

# Confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test_f, y_pred_final)
fig_cm = px.imshow(
    cm, text_auto=True, color_continuous_scale='Blues',
    labels=dict(x='Predicted', y='Actual', color='Count'),
    x=['Loss', 'Win'], y=['Loss', 'Win'],
    title='Final Model: Confusion Matrix'
)
fig_cm.update_layout(template='dsc259r', width=500, height=450)
fig_cm.write_html('assets/confusion_matrix.html', include_plotlyjs='cdn')
fig_cm.show()

## Step 8: Fairness Analysis

In [ ]:
# Step 8: Fairness Analysis
# Does the model perform equally well for major leagues vs minor/regional leagues?

# Group X: Major leagues (LCK, LPL, LEC, LCS) — high-profile, heavily scouted competitions
# Group Y: All other (minor/regional) leagues

major_leagues = ['LCK', 'LPL', 'LEC', 'LCS']
test_leagues = teams_15.loc[test_idx, 'league']
is_major = test_leagues.isin(major_leagues).values

y_test_arr = y_test_f.values
y_pred_arr = y_pred_final

f1_major = f1_score(y_test_arr[is_major], y_pred_arr[is_major])
f1_minor = f1_score(y_test_arr[~is_major], y_pred_arr[~is_major])
obs_f1_diff = f1_major - f1_minor

print(f'F1 (Major leagues — LCK/LPL/LEC/LCS): {f1_major:.4f}  (n={is_major.sum()})')
print(f'F1 (Minor/regional leagues):           {f1_minor:.4f}  (n={(~is_major).sum()})')
print(f'Observed F1 difference (Major − Minor): {obs_f1_diff:.4f}')

# Permutation test
# H0: The model is fair — F1 for major and minor leagues are roughly the same.
# HA: The model is unfair — F1 differs between major and minor leagues.

rng_fair = np.random.default_rng(42)
N_PERM_FAIR = 1000
perm_f1_diffs = np.empty(N_PERM_FAIR)
major_labels = is_major.copy()

for i in range(N_PERM_FAIR):
    rng_fair.shuffle(major_labels)
    f1_a = f1_score(y_test_arr[major_labels], y_pred_arr[major_labels], zero_division=0)
    f1_b = f1_score(y_test_arr[~major_labels], y_pred_arr[~major_labels], zero_division=0)
    perm_f1_diffs[i] = f1_a - f1_b

pval_fairness = np.mean(np.abs(perm_f1_diffs) >= np.abs(obs_f1_diff))

print(f'\nPermutation test p-value: {pval_fairness:.4f}')
print(f'At α = 0.05: {"Reject H0 — model appears unfair" if pval_fairness < 0.05 else "Fail to reject H0 — no evidence of unfairness"}')

# Visualization
fig_fair = go.Figure()
fig_fair.add_trace(go.Histogram(
    x=perm_f1_diffs, marker_color='#1f77b4', opacity=0.7,
    nbinsx=40, name='Permutation Distribution'))
fig_fair.add_vline(
    x=obs_f1_diff, line_dash='dash', line_color='red',
    annotation_text=f'Observed = {obs_f1_diff:.4f}')
fig_fair.update_layout(
    title='Fairness Test: F1 Difference (Major − Minor Leagues)',
    xaxis_title='F1 Difference', yaxis_title='Count',
    template='dsc259r')
fig_fair.write_html('assets/fairness_test.html', include_plotlyjs='cdn')
fig_fair.show()